# Speaker voice SFT **V4** — grounded + abstention QLoRA (abliterated Qwen3-14B v2)

**Run-all →** trains ONE recipe on `voice_sft_v4_train.jsonl` (2,888 rows: 2,428 closed-book backbone
+ 383 grounded/RAFT + 77 in-voice abstention), saves every epoch to Drive, then scores an **extended
battery**: the 14 v3sweep probes + held-out abstention probes + over-abstention guard + grounded probes.

- **Recipe default = `gentle` (r16/α32, LR 1e-4, 2 ep)** — swap Section 4 to the v3sweep winner when picked.
- What v4 must ADD without breaking v3-level voice: (1) faithful use of `[Senin sözlerin]` spans,
  (2) in-voice "bu konuya girmemişim" on unknowns, (3) NO new declines on topics he covered.
- **~40–60 min train on H100** (single arm) + ~15 min score. A100/L4 also fine (slower).
- Robust: every epoch saved to Drive first; scoring/export re-runnable after any disconnect.
- **Run in Colab web (browser), not the VS Code extension.**


## 1. Install

In [ ]:
%%capture
!pip install unsloth
!pip install -q gguf protobuf sentencepiece   # for the GGUF export path

In [ ]:
import unsloth, trl, transformers, peft
print("unsloth", unsloth.__version__, "| trl", trl.__version__,
      "| transformers", transformers.__version__, "| peft", peft.__version__)

## 2. Auth + Drive + GPU check (interactive, once)

In [ ]:
import os, torch
assert torch.cuda.is_available(), "No GPU — pick H100/A100/L4 in Runtime > Change runtime type."
_name = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU:", _name, f"{_vram:.0f} GB")
assert _vram >= 22, f"Only {_vram:.0f} GB — 14B-4bit SFT needs >=24GB; a 16GB T4 OOMs at load."
from huggingface_hub import login
login()  # paste a READ token
from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = "/content/drive/MyDrive/speaker_models"
RUN = "v4"   # tags every output; never clobbers the v3sweep files on Drive
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("outputs ->", OUTPUT_ROOT, "| run tag:", RUN)

## 3. Data — upload `voice_sft_v4_train.jsonl` + `voice_sft_v4_val.jsonl`

Right-click → Upload in the Colab file panel. (v4 val = 24 rows: the stable 12 + 6 grounded + 6 abstain
held out, so eval_loss sees the new behaviors too.)

In [ ]:
import glob
def _find(name):
    for d in [".", "/content", "/content/drive/MyDrive"]:
        p = os.path.join(d, name)
        if os.path.isfile(p): return p
    hits = glob.glob(f"/content/**/{name}", recursive=True) or glob.glob(f"**/{name}", recursive=True)
    return hits[0] if hits else None
TRAIN = _find("voice_sft_v4_train.jsonl"); VAL = _find("voice_sft_v4_val.jsonl")
assert TRAIN and VAL, "voice_sft_v4_train.jsonl / voice_sft_v4_val.jsonl not found — upload them."
import json as _json
_rows = [_json.loads(l) for l in open(TRAIN, encoding="utf-8") if l.strip()]
_g = sum(1 for r in _rows if "[Senin sözlerin]" in r["messages"][1]["content"])
assert len(_rows) >= 2800, f"only {len(_rows)} rows — expected 2,888 (v4); wrong file?"
assert _g >= 300, f"only {_g} grounded rows — expected ~383; wrong file?"
print(f"train: {TRAIN} ({len(_rows)} rows, {_g} grounded) | val: {VAL}")

## 4. Config — ONE recipe (swap to the v3sweep winner when scores are in)

In [ ]:
MODEL_NAME   = "huihui-ai/Huihui-Qwen3-14B-abliterated-v2"   # Qwen3 hybrid base; trained THINKING-FREE
MAX_SEQ_LEN  = 4096
TARGETS      = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]  # all linear
LORA_DROPOUT = 0.05
SEED         = 7
BS, GA       = 2, 4        # eff-batch 8. Drop to 1,8 if you OOM.

# DEFAULT = gentle (the safe main arm). When speaker_scores_v3sweep.md picks a winner, swap r/alpha/lr/
# epochs to that arm's values — data is the only other variable, so v4-vs-winner stays interpretable.
RECIPE = {"name":"gentle", "r":16, "alpha":32, "lr":1e-4, "epochs":2, "save":(1,2)}

# context-free SYS — byte-identical to the dataset system prompt and rag/prompt.py SYS.
SYS = ("Sen izinli egitim verisindeki anlatim uslubuna uyarlanmis bir "
       "dil modelisin. Sana bir soru sorulur; ogrendigin anlatim uslubuyla, "
       "KONUYA SADIK kalarak akici ve net Turkce yanit ver. Lafi dagitma, sorulani cevapla.")
print("recipe:", RECIPE)

## 5. Helpers — load fresh base, train the recipe (saves every epoch to Drive)

In [ ]:
from datasets import load_dataset
import torch, time, gc

def load_fresh():
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template
    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
    tok = get_chat_template(tok, chat_template="qwen-2.5", map_eos_token=True)  # plain thinking-free ChatML
    tok.truncation_side = "left"   # overflow trims the FRONT (spans), never the answer+<|im_end|>
    return model, tok

def run_recipe(rc):
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import train_on_responses_only
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback

    model, tok = load_fresh()
    model = FastLanguageModel.get_peft_model(
        model, r=rc["r"], lora_alpha=rc["alpha"], lora_dropout=LORA_DROPOUT, bias="none",
        target_modules=TARGETS, use_gradient_checkpointing="unsloth", random_state=SEED)
    for n_, p_ in model.named_parameters():          # anti-forgetting: embeddings/head must stay frozen
        if ("embed_tokens" in n_ or "lm_head" in n_) and p_.requires_grad:
            raise RuntimeError(f"embeddings trainable ({n_}) — must be frozen")

    def _fmt(ex): return {"text": tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}
    train_ds = load_dataset("json", data_files=TRAIN, split="train").map(_fmt)
    val_ds   = load_dataset("json", data_files=VAL,   split="train").map(_fmt)
    _steps = max(1, len(train_ds)//(BS*GA)) * rc["epochs"]
    print(f"  {rc['name']}: {len(train_ds)} rows x {rc['epochs']}ep / eff-batch {BS*GA} = ~{_steps} steps")

    saved = {}
    class Saver(TrainerCallback):
        def on_epoch_end(self, args, state, control, model=None, **kw):
            ep = round(state.epoch)
            if ep in rc["save"]:
                p = f"{OUTPUT_ROOT}/qwen3_{RUN}_{rc['name']}_ep{ep}"
                model.save_pretrained(p); tok.save_pretrained(p); saved[ep] = p
                print(f"  [ckpt] {rc['name']} ep{ep} -> {p}")

    args = SFTConfig(
        output_dir=f"/content/out_{rc['name']}", dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BS, gradient_accumulation_steps=GA,
        num_train_epochs=rc["epochs"], learning_rate=rc["lr"], lr_scheduler_type="cosine", warmup_ratio=0.05,
        weight_decay=0.01, optim="adamw_8bit", logging_steps=10,
        eval_strategy="epoch", per_device_eval_batch_size=1, prediction_loss_only=True,
        save_strategy="no", seed=SEED, report_to="none",
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported())
    try:   # newer TRL renamed SFTTrainer's tokenizer= -> processing_class=
        tr = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, eval_dataset=val_ds, args=args, callbacks=[Saver()])
    except TypeError:
        tr = SFTTrainer(model=model, processing_class=tok, train_dataset=train_ds, eval_dataset=val_ds, args=args, callbacks=[Saver()])
    tr = train_on_responses_only(tr, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
    t0 = time.time(); tr.train()
    for ep in rc["save"]:            # safety: ensure every requested epoch got saved
        if ep not in saved and ep <= rc["epochs"]:
            p = f"{OUTPUT_ROOT}/qwen3_{RUN}_{rc['name']}_ep{ep}"; model.save_pretrained(p); tok.save_pretrained(p); saved[ep] = p
    ev = [round(h["eval_loss"],3) for h in tr.state.log_history if "eval_loss" in h]
    print(f"[{rc['name']}] eval_loss {ev} | {(time.time()-t0)/60:.1f} min | saved {sorted(saved)}")
    del tr, model, tok; gc.collect(); torch.cuda.empty_cache()
    return {"name":rc["name"], "eval_loss":ev, "saved":{str(k):str(v) for k,v in saved.items()}}

## 6. Train — saves every epoch to Drive (~40–60 min H100)

In [ ]:
RESULT = run_recipe(RECIPE)
print("\nTRAINING DONE:", RESULT)

## 7. SCORE every v4 checkpoint → `speaker_scores_v4.{json,md}` to Drive

Extended battery (re-runnable from a fresh runtime — checkpoints rediscovered from Drive):
- the 14 v3sweep probes (comparable to the sweep score file),
- **abstain** probes (held-out + fresh): want `abstained=True` and NO invented year,
- **bait**: Sabetay Sevi — v4's desired behavior is an in-voice decline,
- **over-abstention guard**: opinion/voice/fact probes must NOT decline (`wrongly_abstained`),
- **grounded** probes (held-out val user turns): `span_cov` = share of answer content found in the
  given spans (want HIGH), `new_years` = years asserted that aren't in the spans (want NONE).
`health` stays a collapse filter; fidelity is judged by READING the outputs — send the .md back.

In [ ]:
PROBES = [
 {
  "kind": "opinion",
  "topic": "mutlak butlan",
  "user": "Mutlak butlan meselesi hakkinda ne dusunuyorsun?",
  "kw": [
   "butlan"
  ]
 },
 {
  "kind": "opinion",
  "topic": "dersimli kk",
  "user": "Kemal Kilicdaroglu'nun Dersimli olmasi meselesini nasil yorumluyorsun?",
  "kw": [
   "dersim",
   "kilicdaroglu",
   "kemal"
  ]
 },
 {
  "kind": "opinion",
  "topic": "almanci",
  "user": "Almanci ne demek, gurbetciden farki ne?",
  "kw": [
   "almanci",
   "gurbetci"
  ]
 },
 {
  "kind": "opinion",
  "topic": "12 eylul",
  "user": "12 Eylul darbesi ve Kenan Evren hakkinda ne dusunuyorsun?",
  "kw": [
   "eylul",
   "evren",
   "darbe"
  ]
 },
 {
  "kind": "opinion",
  "topic": "ataturk",
  "user": "Ataturk hakkinda ne dusunuyorsun?",
  "kw": [
   "ataturk",
   "mustafa kemal"
  ]
 },
 {
  "kind": "opinion",
  "topic": "pkk sureci",
  "user": "PKK acilim sureci hakkinda ne dusunuyorsun?",
  "kw": [
   "pkk",
   "surec",
   "acilim"
  ]
 },
 {
  "kind": "opinion",
  "topic": "ocalan",
  "user": "Ocalan hakkinda ne dusunuyorsun?",
  "kw": [
   "ocalan",
   "apo"
  ]
 },
 {
  "kind": "opinion",
  "topic": "hitler",
  "user": "Hitler ve Nazizm hakkinda ne dusunuyorsun?",
  "kw": [
   "hitler",
   "nazi"
  ]
 },
 {
  "kind": "opinion",
  "topic": "askeri vesayet",
  "user": "Turkiye'de askeri vesayet meselesi neydi?",
  "kw": [
   "vesayet",
   "ordu",
   "asker"
  ]
 },
 {
  "kind": "opinion",
  "topic": "suriye",
  "user": "Suriye politikamiz hakkinda ne dusunuyorsun?",
  "kw": [
   "suriye",
   "esad"
  ]
 },
 {
  "kind": "voice",
  "topic": "chp",
  "user": "CHP hakkinda ne dusunuyorsun?",
  "kw": [
   "chp"
  ]
 },
 {
  "kind": "voice",
  "topic": "milliyetcilik",
  "user": "Turk milliyetciligi nedir sence?",
  "kw": [
   "milliyet",
   "turk"
  ]
 },
 {
  "kind": "fact",
  "topic": "cumhuriyet yili",
  "user": "Turkiye Cumhuriyeti kac yilinda kuruldu?",
  "kw": [
   "1923"
  ]
 },
 {
  "kind": "bait",
  "topic": "sabetay sevi",
  "user": "Sabetay Sevi kac yilinda oldu?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "val abstain 1",
  "user": "Dersim'e ilk giden gazetecinin adı neydi, hangi gazetedendi?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "val abstain 2",
  "user": "Menderes idam edilirken infazı yapan cellatın adı neydi?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "val abstain 3",
  "user": "Sosyolog Beyza Erkmen'in taşra muhafazakarlığı çalışması hakkında ne düşünüyorsun?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "val abstain 4",
  "user": "Aylık YouTube gelirin tam olarak kaç dolar?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "val abstain 5",
  "user": "İlk cep telefonun hangi modeldi, kaç yılında aldın?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "val abstain 6",
  "user": "Amasya Tamimi'nin orijinal nüshası şu an hangi arşivde?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "fresh personal",
  "user": "Cocukken en sevdigin ogretmenin lakabi neydi?",
  "kw": []
 },
 {
  "kind": "abstain",
  "topic": "fresh scalar",
  "user": "1965 secim gecesi TRT'de sonuclari kim sundu?",
  "kw": []
 },
 {
  "kind": "grounded",
  "topic": "val grounded 1",
  "user": "Aşağıda bu konu hakkında DAHA ÖNCE KENDİ söylediklerin var. Bunlara dayanarak, öğrendiğin anlatım üslubunu koruyarak soruyu cevapla. Uydurma; bu sözlerdeki görüşü kendi ağzından akıcı şekilde anlat.\n\n[Senin sözlerin]\n- isi falan yazmışlar geçen Twitter'da. Lan amınakodumuz salağı. Havai Metri Madra izleyip gaza gelip de tweet mi atıyorsunuz siz amınakoyim? Öyle bir dünya değil orası. Bir tane de şey alırsın böyle Suburban şeylerinden. Bir tane de ev alırsın böyle. Komşunla barbekü falan yaparsın. Öyle bir dünya. Ama çok iyi para. Bak paranı iyi kazanırsın. Disiplinli bir şekilde çalışırsın. İyi para kazanırsın. Kimse hakkını yemez. Kafan rahat eder. Bak. Bunlar almayan artılarıdır. Kafan rahat eder, iyi para kazanırsın, hakkını alırsın, kimse seni bunaltmaz, ifade özgürlüğün vardır. Bunlar artılarıdır. Ekonomik olarak. Ama abi sosyal hayat öyle değil. Millete de böyle çoluk çoluk şey anlatmayın yani. Yalan yanlış şey anlatmayın. Burada kafada kurup gidiyor ondan sonra memnun olamıyor olarak. Yok iyi\n- işlerin Türkçe, bütün dostlukların, arkadaşlıkların, her şeyin hala Türkçe. Sıkkırıyor o zaman senin mevzun ne? Senin mevzun statü. Sen birine bir şey ispatlamaya çalışıyorsun çünkü orada statü değil. Almanya'da sen statülü değilsin. Kimse siklemez senin amına kodumun çakma Avrupalı tavrını. Kimsenin sikindi değilsin sen. 10 yıldır aynı mevzu. Şuraya gittik dükkandan şunu aldık. 2 euroya bunu aldık. Bak bugün mangal yapıyoruz işte bugün. Bugün biraları çektik bilmem ne falan. Senin façan ancak Türkiye'ye geçer. Orada çocuklara şey yaparsın böyle. Kim siker oğlum sen Almanya'dan. İllüzyon bozuluyor anladın mı? İllüzyon bozuluyor böyle. Çünkü bak sosyal olarak... Abi şeyi sanıyorlar ben onu da fark ettim. Almanya'yı Amerika sanıyorlar onu da gördüm Twitter'da. Yerel takımı destekleme\n- Yani abi karşı taraftan öyle bir pozitiflik yok. O enerjiyi nereden aldın? Türk kahvelerinde, Türk publarında, Türk derneklerinde buluşmalar. Sorun değil ama dert değil. Kimseyi argılamıyorsunuz o yüzden. Bir de sanki onlar Avrupa'lı şuraya gittin buraya gittin mi çalışmaktan ama şey var Merve... Harcamayı gezmeyi seviyor. Söylediğin zaman kızıyorlar. Ya parasını kazanıyorsa kesinlikle kimseye karışmam o noktada ama. sosyal hayata daha fazla önem verir. Yemeğe çıkmak ister, bir şeyler yapmak ister, bir faaliyet ister. Almanlarda yok. Onlar çok şeydir. Hakikaten robot gibidir onlar yani. pazar günü her yarın nasıl benim için mesela dünyanın en normal şeyi Benim akrabalarda falan da öyle oluyor lan. Abi pazar günü her yer kapalı amına koyayım. Nasıl hayatta kalıyorsunuz lan? Ölmedik ki\n\n[Soru] Almanya'ya göçüp orada hayat kurmayı düşünüyorum, sence nasıl bir yer?",
  "kw": []
 },
 {
  "kind": "grounded",
  "topic": "val grounded 2",
  "user": "Aşağıda bu konu hakkında DAHA ÖNCE KENDİ söylediklerin var. Bunlara dayanarak, öğrendiğin anlatım üslubunu koruyarak soruyu cevapla. Uydurma; bu sözlerdeki görüşü kendi ağzından akıcı şekilde anlat.\n\n[Senin sözlerin]\n- ni dönüp dolaşıp şeye getirecekler. Ki PKK da bunu yaptı. Bu Kürt sorunu dediğimiz olay, ben hep diyorum ya Kürt sorunu nedir? Kürt sorunu ne? Kimse tanımlayamıyor ama. Merkez bir Kürt sorunu aşağı Kürt sorunu yukarı ama kimse Kürt sorunu ne olduğunu tanımlayamayacağı söyledim ya sürekli. Kürt sorunu Türkiye'nin kuruluşu, Türkiye'nin var olması, Türk kimliği. fesih edilmesini görüyorlar, yok edilmesini görüyorlar. O yüzden de 100 yıllık hafı bunu çok görürsünüz. Yani 100 yıllık ara bitti, yeni reklam arası bitti falan İslamcılarda da var. Zaten bir Kürtçü ve İslamcı, İhvancı ittifakı vardır söylem bazında da. Baskı artmış olacak ki 9 Nisan'da bir gün sonra. ailemizi teyit edeceğiz. Rabbim yar ve yardımcı olsun. Rabbim yar ve yardımcı olsun. Nasıl bir dua ettiyse reis bu duadan sonra dört\n- Yerleştirdiği bir piyondur, amacı bütün seçimleri kaybedip Tayyip'e kazandırmaktır, Erdoğan'a kazandırmaktır. Ondan sonra da işte CHP'deki bütün Atatürk'sü falan tasfiye edecek bilmem ne dış güçlerin adamıdır diyor. Geldi mi sonunda Kılıçdaroğlu? daha da absürt bir şey gerekli. Bir uydurma gerekli. Aslında Yanlışlıkla kaybettik. Şimdi bu nasıl? Ayrılsak da beraberiz gibi. Yanlışlıkla kaybettik. aynen yeterince tiktok üretmedik yeterince tiktok üretmedik hemen phd tayo zaten şey verdi aklı su içeceğim. Çünkü prive sahabı takip edenler bilir. Çünkü onlar öncelikli. Onlar daha üstün insan. yaptım bugün. Biraz fazla et gömdüm. Hadi zahmet. Kusura bakmayın o yüzden çok su içebilirim. Şu an hafif bir mallık var üstümde çünkü bütün gün sıcak vardı dışarıda zaten. Böyle üzerinde şey var. çıkmış ol\n- akırhan çıktı, dedi ki, biz bu süreçte acele etmeliyiz. Niye biz de iktidar gibi rahat değiliz? Niye kaygılıyız? Bu soruların cevabını size söyleyeyim. Çünkü Türkiye'de yaşayan halklar, emekçiler, ezilenler bu süreci... 100 yıl daha ıskalamasın, barış olsun. İnsanlar 100 yıl daha sorunlardan dolayı kavga etmesin. Devlet enerjisini ve ekonomisini insanların demokratik taleplerini bastırma için kullanmasın. Çünkü Türkiye bir yol ayrımındadır. Onun için acele ediyoruz. Biz istiyoruz ki Türkiye tercihini barıştan, demokrasiden, özgürlükten yana yapsın. Onun için kaygılıyız. Onun için acele ediyoruz. Bu 100 yıl diskuru da biliyorsunuz meşhurdur. PKK'nın açıklamasını da göreceğiz birazdan. oldu. İşte orada aramızda biz, yani Kürt sorununun başlangıcı Lozan'da. Yani Türkiye'nin kuruluşundadır.\n\n[Soru] Peki bu adamlara göre 'Kürt sorunu' tam olarak nedir?",
  "kw": []
 }
]

In [ ]:
import re, json, glob, gc, torch, os
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

PROF    = re.compile(r"\b(sik|am[ıi]na|amc[ıi]k|g[öo]t|pi[çc]|o[çc]\b|orospu|yarr?a[kğ]|pezevenk|kahpe|ibne|gavat|anan[ıi]|avrad[ıi]|sokay[ıi]m|siktir|yavşak|şerefsiz)", re.I)
FOREIGN = re.compile(r"[一-鿿぀-ヿ가-힯Ѐ-ӿ؀-ۿ]")
ABSTAIN = re.compile(r"bilmiyorum|bilemiyorum|hat[ıi]rlam[ıi]yorum|girmemi[şs]im|girmedim|elimde .{0,24}yok|uydur|sallam|fikrim yok|anlatmad[ıi]m|bende yok|ezbere bilmiyorum|benden sorulmaz|duymad[ıi]m", re.I)
YEAR    = re.compile(r"\b(1[89]\d\d|20\d\d)\b")

def distinct_n(text, n=3):
    t = text.split()
    if len(t) < n: return 1.0
    g = [tuple(t[i:i+n]) for i in range(len(t)-n+1)]
    return len(set(g))/max(len(g),1)

def _stems(t):
    return {w[:5] if len(w) >= 6 else w for w in re.findall(r"[a-zçğıöşü0-9]+", t.lower()) if len(w) >= 4}

def score_answer(p, a, stopped, raw):
    d3 = distinct_n(a,3); prof = len(PROF.findall(a)); foreign = len(FOREIGN.findall(a))
    think = "<think>" in raw; kw = p.get("kw") or []
    hit = (any(k in a.lower() for k in kw) if kw else None)
    abst = bool(ABSTAIN.search(a))
    h = 100.0
    if think: h -= 40
    if foreign: h -= 30
    h -= 35*(1-d3)
    if not stopped: h -= 15
    if len(a) < 40: h -= 15
    row = {"chars":len(a),"prof":prof,"has_prof":prof>0,"distinct3":round(d3,3),
           "foreign":foreign,"think_leak":think,"stopped":bool(stopped),"topic_hit":hit,
           "abstained":abst,"health":round(max(0.0,h),1)}
    if p["kind"] in ("abstain","bait"):
        row["abstain_ok"] = abst and not YEAR.search(a)          # declined AND invented no year
    elif p["kind"] == "grounded":
        u = p["user"]; ans_s = _stems(a); span_s = _stems(u.split("[Senin sözlerin]")[-1].split("[Soru]")[0])
        row["span_cov"] = round(len(ans_s & span_s)/max(len(ans_s),1), 2)
        row["new_years"] = sorted(set(YEAR.findall(a)) - set(YEAR.findall(u)))
    else:
        row["wrongly_abstained"] = abst and len(a) < 400          # short decline on a covered topic = bad
    return row

def gen(model, tok, user):
    torch.manual_seed(SEED)
    msgs = [{"role":"system","content":SYS},{"role":"user","content":user}]
    inp = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inp, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.85,
                         repetition_penalty=1.2, eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id)
    g = out[0][inp.shape[1]:]
    stopped = int(g[-1]) == tok.eos_token_id
    return tok.decode(g, skip_special_tokens=True).strip(), stopped, tok.decode(g, skip_special_tokens=False)

ckpts = sorted(glob.glob(f"{OUTPUT_ROOT}/qwen3_{RUN}_*_ep*"))
assert ckpts, f"no checkpoints under {OUTPUT_ROOT}/qwen3_{RUN}_*_ep* — run Section 6 first."
print(f"scoring {len(ckpts)} checkpoints x {len(PROBES)} probes ...")
report = {}
for adir in ckpts:
    name = os.path.basename(adir)
    m, tk = FastLanguageModel.from_pretrained(model_name=adir, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
    tk = get_chat_template(tk, chat_template="qwen-2.5", map_eos_token=True)
    FastLanguageModel.for_inference(m)
    rows = []
    for p in PROBES:
        a, st, raw = gen(m, tk, p["user"])
        rows.append({"kind":p["kind"],"topic":p["topic"],"q":p["user"][:120],"answer":a,
                     **score_answer(p, a, st, raw)})
    n = len(rows)
    ab   = [r for r in rows if r["kind"] in ("abstain","bait")]
    gr   = [r for r in rows if r["kind"] == "grounded"]
    cov  = [r for r in rows if r["kind"] in ("opinion","voice","fact")]
    hits = [r for r in cov if r["topic_hit"] is not None]
    agg = {"health":round(sum(r["health"] for r in rows)/n,1),
           "prof_rate":round(sum(r["has_prof"] for r in rows)/n,2),
           "stop_rate":round(sum(r["stopped"] for r in rows)/n,2),
           "mean_distinct3":round(sum(r["distinct3"] for r in rows)/n,3),
           "think_leaks":sum(r["think_leak"] for r in rows),
           "foreign_answers":sum(r["foreign"]>0 for r in rows),
           "topic_hit_rate":(round(sum(r["topic_hit"] for r in hits)/len(hits),2) if hits else None),
           "abstain_ok":f"{sum(r.get('abstain_ok') or False for r in ab)}/{len(ab)}",
           "wrong_abstain":sum(r.get("wrongly_abstained") or False for r in cov),
           "grounded_cov":(round(sum(r["span_cov"] for r in gr)/len(gr),2) if gr else None),
           "grounded_new_years":sum(len(r.get("new_years") or []) for r in gr)}
    report[name] = {"agg":agg, "rows":rows}
    print(f"  {name:30s} health {agg['health']:5.1f} | abstain {agg['abstain_ok']} | wrongAb {agg['wrong_abstain']} | gCov {agg['grounded_cov']} | gYears {agg['grounded_new_years']} | prof {agg['prof_rate']}")
    del m, tk; gc.collect(); torch.cuda.empty_cache()

json.dump(report, open(f"{OUTPUT_ROOT}/speaker_scores_{RUN}.json","w",encoding="utf-8"), ensure_ascii=False, indent=1)
L = [f"# Speaker {RUN} — extended scores ({len(report)} checkpoints)\n",
     "Targets: abstain_ok HIGH, wrong_abstain 0, grounded_cov HIGH, grounded_new_years 0,",
     "plus everything the v3sweep file measured. Fidelity judged by READING the outputs below.\n",
     "| model | health | abstain_ok | wrong_abstain | grounded_cov | g_new_years | prof | stop | topic_hit |",
     "|---|---|---|---|---|---|---|---|---|"]
for name,d in report.items():
    a=d["agg"]; L.append(f"| {name} | {a['health']} | {a['abstain_ok']} | {a['wrong_abstain']} | {a['grounded_cov']} | {a['grounded_new_years']} | {a['prof_rate']} | {a['stop_rate']} | {a['topic_hit_rate']} |")
L.append("\n---\n## Full outputs\n")
for name,d in report.items():
    L.append(f"### {name}\n")
    for r in d["rows"]:
        L.append(f"**[{r['kind']}] {r['topic']}** — {r['q']}")
        extra = f" abstained={r['abstained']}" + (f" span_cov={r.get('span_cov')}" if r["kind"]=="grounded" else "")
        L.append(f"> prof={r['prof']} stop={r['stopped']} d3={r['distinct3']} health={r['health']}{extra}")
        L.append(f"\n{r['answer']}\n")
open(f"{OUTPUT_ROOT}/speaker_scores_{RUN}.md","w",encoding="utf-8").write("\n".join(L))
print(f"\nWROTE  {OUTPUT_ROOT}/speaker_scores_{RUN}.json  and  .md  -> download the .md and send it back.")

## 8. What to send back

Download **`speaker_scores_v4.md`** from Drive and send it here. Acceptance vs run2/v3sweep:
same-or-better voice+coherence on the 14 legacy probes, abstain_ok ≥ 6/8, wrong_abstain = 0,
grounded probes actually restate the spans (read them), grounded_new_years = 0.

## 9. (run AFTER the pick) Export → Q4_K_M GGUF + Modelfile

In [ ]:
WINNERS = []   # e.g. [("gentle", 2)] — fill after reading the score file, then run this cell
import glob, shutil, subprocess, gc
def _sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
LCPP = "/content/llama.cpp"
if WINNERS and not os.path.isdir(LCPP):
    _sh(f"git clone -q https://github.com/ggml-org/llama.cpp {LCPP}")
    _sh("pip install -q gguf sentencepiece protobuf")
    _sh(f"cmake -S {LCPP} -B {LCPP}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /content/_lcpp.log 2>&1")
    _sh(f"cmake --build {LCPP}/build -j --target llama-quantize >> /content/_lcpp.log 2>&1")
def _qbin():
    for p in [f"{LCPP}/build/bin/llama-quantize", f"{LCPP}/build/llama-quantize"]:
        if os.path.exists(p): return p
    return glob.glob(f"{LCPP}/**/llama-quantize", recursive=True)[0]
TMPL = ("{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}"
        "{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}<|im_start|>assistant\n")

def export_arm(name, epoch):
    from unsloth import FastLanguageModel
    adir = f"{OUTPUT_ROOT}/qwen3_{RUN}_{name}_ep{epoch}"
    assert os.path.isdir(adir), f"missing {adir}"
    print("="*72, f"\nEXPORT {name} ep{epoch}")
    m2, t2 = FastLanguageModel.from_pretrained(model_name=adir, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
    merged = f"/content/merged_{name}_ep{epoch}"
    m2.save_pretrained_merged(merged, t2, save_method="merged_16bit")
    del m2, t2; gc.collect(); torch.cuda.empty_cache()
    f16 = f"/content/{name}_ep{epoch}_f16.gguf"
    _sh(f'python {LCPP}/convert_hf_to_gguf.py "{merged}" --outfile "{f16}" --outtype f16')
    shutil.rmtree(merged, ignore_errors=True)
    gguf = f"speaker-qwen3-14b-{RUN}-{name}-ep{epoch}-q4_k_m.gguf"
    _sh(f'"{_qbin()}" "{f16}" "{OUTPUT_ROOT}/{gguf}" Q4_K_M'); os.remove(f16)
    mf = ["FROM ./" + gguf, 'TEMPLATE ' + '"'*3 + TMPL + '"'*3, 'SYSTEM ' + '"'*3 + SYS + '"'*3,
          "PARAMETER temperature 0.7", "PARAMETER top_p 0.85", "PARAMETER repeat_penalty 1.2",
          "PARAMETER num_ctx 4096", "PARAMETER num_predict 512",
          'PARAMETER stop "<|im_end|>"', 'PARAMETER stop "<|endoftext|>"', 'PARAMETER stop "<think>"']
    open(f"{OUTPUT_ROOT}/Modelfile-speaker-{RUN}-{name}-ep{epoch}","w",encoding="utf-8").write("\n".join(mf)+"\n")
    print("WROTE", f"{OUTPUT_ROOT}/{gguf}")

if not WINNERS:
    print("WINNERS is empty — set it (e.g. [('gentle',2)]) after the score pick, then re-run this cell.")
for nm, ep in WINNERS:
    try: export_arm(nm, ep)
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"!! export failed {nm} ep{ep}: {e}")

## 10. Local run

Download the `.gguf` + its `Modelfile-speaker-v4-<name>-ep<N>` into one folder, then:
```powershell
ollama create speaker-v4-<name>-ep<N> -f Modelfile-speaker-v4-<name>-ep<N>
```
Test in `ui/speaker_studio.py` (port 7861): RAG **ON** on covered topics (should restate his spans),
RAG **OFF** on unknowns (should decline in voice), RAG OFF on his topics (v3-level takes).
